# Gate 2 Task 3: Training Folds and Target Handling

Owner: Rachel Cheung · Issue #40 · Approach 1 training is issue #44 (Gate 3).

This notebook is self-contained: it imports nothing from `scripts/`. It
1. builds the 5 cross-validation folds **inside the training partition only** and saves them,
2. recomputes the September target-informed signals **within the training folds**,
3. decides the target transformation and tests the inverse rule, and
4. writes and smoke-tests the Approach 1 (regularized GLM) configuration.

**Boundaries:** only training targets are read; validation and test rows are skipped before their `loss` is parsed. No validation score is computed, and no GLM is fit on real targets. Run the cells top to bottom.

## 1 Setup and frozen split
Finds the project folder, checks the source and partition manifest against their recorded hashes, and loads the claim IDs for each partition. No targets are read here.

In [1]:
import csv
import hashlib
import json
import sys
from datetime import date
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import skew
from sklearn.model_selection import KFold

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "allstate_claims_data.csv").is_file())
SOURCE = ROOT / "data" / "allstate_claims_data.csv"
GATE1 = ROOT / "artifacts" / "october" / "gate 1"
GATE2 = ROOT / "artifacts" / "october" / "gate 2"
FOLD_DIR = GATE2 / "cv_folds"
FOLD_FILE = FOLD_DIR / "october_cv_folds.csv"
FOLD_METADATA = FOLD_DIR / "october_cv_folds_metadata.json"
CONFIG_FILE = GATE2 / "approach1_glm_config.json"

N_FOLDS, FOLD_SEED, RARE_THRESHOLD = 5, 42, 30
CATEGORICAL_COLUMNS = [f"cat{i}" for i in range(1, 117)]
CONTINUOUS_COLUMNS = [f"cont{i}" for i in range(1, 15)]
FEATURE_COLUMNS = CATEGORICAL_COLUMNS + CONTINUOUS_COLUMNS

results = []

def check(name, ok):
    results.append((name, bool(ok)))
    print(f"{'PASS' if ok else 'FAIL'}: {name}")

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

print("Python:", sys.version.split()[0])
for package in ["pandas", "numpy", "scipy", "scikit-learn"]:
    print(f"{package}: {version(package)}")

Python: 3.13.16
pandas: 2.3.2
numpy: 2.3.3
scipy: 1.18.1
scikit-learn: 1.7.2


In [2]:
metadata = json.loads((GATE1 / "october_split_metadata.json").read_text())
manifest_path = GATE1 / "october_partition_manifest.csv"

source_sha = sha256(SOURCE)
manifest_sha = sha256(manifest_path)
check("Source matches the frozen hash", source_sha == metadata["source"]["sha256"])
check("Partition manifest matches its recorded hash",
      manifest_sha == metadata["artifacts_sha256"]["october_partition_manifest.csv"])

manifest = pd.read_csv(manifest_path)
train_ids = np.sort(manifest.loc[manifest.partition.eq("train"), "id"].to_numpy())
valid_ids = set(manifest.loc[manifest.partition.eq("validation"), "id"])
test_ids = set(manifest.loc[manifest.partition.eq("test"), "id"])
check("Partition counts match the split (141,238 / 28,248 / 18,832)",
      (len(train_ids), len(valid_ids), len(test_ids)) == (141238, 28248, 18832))

PASS: Source matches the frozen hash
PASS: Partition manifest matches its recorded hash
PASS: Partition counts match the split (141,238 / 28,248 / 18,832)


## 2 Build the cross-validation folds

Team policy `OCT-G1-T4-PROPOSAL-v1`: five shuffled folds, seed 42, built from the training IDs sorted by `id`. Assignment uses **IDs only**, never `loss` or predictors. Folds are not stratified on the target; Section 4 checks that they came out balanced.

Each fold is held out once. The other four folds (about 113,000 claims) are the **fit portion**, and every learned value is refit on it.

In [3]:
def make_cv_folds(ids):
    ids = np.array(sorted(int(i) for i in ids), dtype=np.int64)
    fold = np.empty(len(ids), dtype=np.int64)
    splitter = KFold(n_splits=N_FOLDS, shuffle=True, random_state=FOLD_SEED)
    for number, (_, held_out) in enumerate(splitter.split(ids)):
        fold[held_out] = number
    return pd.DataFrame({"id": ids, "fold": fold})

def iter_folds(folds):
    for number in range(N_FOLDS):
        held_out = folds.loc[folds.fold.eq(number), "id"].to_numpy()
        fit = folds.loc[folds.fold.ne(number), "id"].to_numpy()
        yield number, fit, held_out

folds = make_cv_folds(train_ids)
counts = folds.fold.value_counts().sort_index()

check("Every training claim assigned exactly once", folds.id.is_unique and set(folds.id) == set(train_ids))
check("No validation or test claim in any fold", set(folds.id).isdisjoint(valid_ids | test_ids))
check("Five folds, sizes differ by at most one claim", len(counts) == N_FOLDS and counts.max() - counts.min() <= 1)
check("Same folds regardless of input order",
      make_cv_folds(np.random.default_rng(7).permutation(train_ids)).equals(folds))

counts.rename("held-out claims").to_frame()

PASS: Every training claim assigned exactly once
PASS: No validation or test claim in any fold
PASS: Five folds, sizes differ by at most one claim
PASS: Same folds regardless of input order


,held-out claims
fold,
0,28248
1,28248
2,28248
3,28247
4,28247


**Save the folds.** The file holds only `id` and `fold`. If it already exists, it is never overwritten: the rebuilt folds must match it exactly.

In [4]:
FOLD_DIR.mkdir(parents=True, exist_ok=True)

if FOLD_FILE.exists():
    check("Saved fold file matches rebuilt folds", pd.read_csv(FOLD_FILE).equals(folds))
else:
    folds.to_csv(FOLD_FILE, index=False, lineterminator="\n")
    FOLD_METADATA.write_text(json.dumps({
        "fold_version": "october-cv-v1",
        "status": "PROPOSED; persisted for Gate 2 verification",
        "created": date.today().isoformat(),
        "owner": "Rachel Cheung",
        "policy": "OCT-G1-T4-PROPOSAL-v1",
        "method": "sklearn KFold(n_splits=5, shuffle=True, random_state=42) over training IDs sorted ascending",
        "uses_target_or_predictors": False,
        "held_out_counts": {str(k): int(v) for k, v in counts.items()},
        "training_claims": len(folds),
        "manifest_id": metadata["split"]["version"],
        "manifest_sha256": manifest_sha,
        "source_sha256": source_sha,
        "fold_file_sha256": sha256(FOLD_FILE),
        "created_by": "notebooks/gate2_training_folds.ipynb",
        "environment": {"python": sys.version.split()[0],
                        **{p: version(p) for p in ["pandas", "numpy", "scipy", "scikit-learn"]}},
        "assumption": "Claims are exchangeable; no confirmed group or time key exists (open Advisor question).",
    }, indent=2) + "\n")

fold_meta = json.loads(FOLD_METADATA.read_text())
check("Fold file matches the hash in its metadata", sha256(FOLD_FILE) == fold_meta["fold_file_sha256"])
print("Fold file SHA-256:", sha256(FOLD_FILE))

PASS: Fold file matches the hash in its metadata
Fold file SHA-256: 82fb7f3d9899af71a61f3e3f1b0117f88ac97b0255070253fd8190ec2c7bfcd1


## 3 Load training targets only
Every non-training row is skipped **before** its `loss` is converted, so validation and test targets are never parsed.

In [5]:
wanted = set(int(i) for i in train_ids)
records = []
with SOURCE.open(newline="") as handle:
    for row in csv.DictReader(handle):
        claim = int(row["id"])
        if claim in wanted:
            records.append((claim, float(row["loss"])))
loss = pd.Series(dict(records), name="loss").sort_index()

check("Targets cover exactly the training claims", set(loss.index) == wanted and len(loss) == len(wanted))
check("No validation or test target loaded", set(loss.index).isdisjoint(valid_ids | test_ids))
check("All training losses finite and positive", np.isfinite(loss).all() and (loss > 0).all())

PASS: Targets cover exactly the training claims
PASS: No validation or test target loaded
PASS: All training losses finite and positive


## 4 Recompute September's target-informed signals within the folds

September computed these on the **full file**, including rows now in validation and test, so October cannot reuse them as fitted values. Each is recomputed on every fold's fit portion and on the full training partition.

| September signal | Full-file value | Where October uses it |
| :--- | :--- | :--- |
| Mean and median `loss` | 3,037.34 / 2,115.57 | Constant baselines in every fold |
| p95 high-loss threshold | 8,508.54 | High-loss error slice (Gate 3) |
| Skewness of `loss` / `log1p(loss)` | 3.795 / 0.097 | Target-transformation decision |
| Pearson with `loss`: cont2, cont7, cont3 | 0.142 / 0.120 / 0.111 | GLM hypothesis (weak linear signal) |

**Not recomputed, because no October decision uses them:** categorical level-to-target differences (FND-003; target encoding is out of scope) and the FND-008 binned-pattern labels (rule never approved). The rare-level threshold uses category counts, not the target, and is refit inside each pipeline.

In [6]:
def target_signals(values):
    v = values.to_numpy()
    return {"claims": len(v), "mean": v.mean(), "median": np.median(v), "p95": np.quantile(v, 0.95),
            "skew_raw": skew(v), "skew_log1p": skew(np.log1p(v))}

rows = {f"fold {k} fit portion": target_signals(loss.loc[fit]) for k, fit, _ in iter_folds(folds)}
rows["all training"] = target_signals(loss)
signals = pd.DataFrame(rows).T
signals.loc["September full file"] = {"claims": 188318, "mean": 3037.34, "median": 2115.57,
                                      "p95": 8508.54, "skew_raw": 3.795, "skew_log1p": 0.097}

fit_rows = signals.iloc[:N_FOLDS]
check("Skew holds in every fit portion (raw > 3, log1p < 0.2)",
      (fit_rows.skew_raw > 3).all() and (fit_rows.skew_log1p.abs() < 0.2).all())
check("Fold medians within 1% of the training median",
      (fit_rows["median"] / signals.loc["all training", "median"] - 1).abs().max() < 0.01)
signals.round({"mean": 2, "median": 2, "p95": 2, "skew_raw": 3, "skew_log1p": 3})

PASS: Skew holds in every fit portion (raw > 3, log1p < 0.2)
PASS: Fold medians within 1% of the training median


,claims,mean,median,p95,skew_raw,skew_log1p
fold 0 fit portion,112990.0,3036.53,2117.10,8495.73,4.270,0.101
fold 1 fit portion,112990.0,3032.40,2111.07,8493.97,3.756,0.098
fold 2 fit portion,112990.0,3046.47,2122.61,8546.60,4.240,0.100
fold 3 fit portion,112991.0,3039.13,2118.68,8525.67,3.653,0.103
fold 4 fit portion,112991.0,3036.96,2113.89,8502.48,4.096,0.107
all training,141238.0,3038.30,2116.66,8511.92,4.006,0.102
September full file,188318.0,3037.34,2115.57,8508.54,3.795,0.097


In [7]:
support = {}
for k, fit, held_out in iter_folds(folds):
    threshold = np.quantile(loss.loc[fit], 0.95)
    support[f"fold {k}"] = {"fit-portion p95": threshold,
                            "held-out claims above it": int((loss.loc[held_out] > threshold).sum()),
                            "held-out claims": len(held_out)}
support = pd.DataFrame(support).T

check("Every held-out fold has at least 1,000 high-loss claims", (support["held-out claims above it"] >= 1000).all())
print("Training p95 for Gate 3 high-loss slices:", round(np.quantile(loss, 0.95), 2))
support.round(2)

PASS: Every held-out fold has at least 1,000 high-loss claims
Training p95 for Gate 3 high-loss slices: 8511.92


,fit-portion p95,held-out claims above it,held-out claims
fold 0,8495.73,1461.0,28248.0
fold 1,8493.97,1464.0,28248.0
fold 2,8546.60,1334.0,28248.0
fold 3,8525.67,1379.0,28247.0
fold 4,8502.48,1434.0,28247.0


In [8]:
cont = pd.concat(chunk[chunk.id.isin(wanted)] for chunk in
                 pd.read_csv(SOURCE, usecols=["id", "cont2", "cont7", "cont3"], chunksize=50_000)).set_index("id")
cont = cont.loc[loss.index]

corr = pd.DataFrame({f"fold {k} fit portion": cont.loc[fit].corrwith(loss.loc[fit])
                     for k, fit, _ in iter_folds(folds)}).T
check("cont2, cont7, cont3 stay weak and positive in every fold (0 < r < 0.2)", ((corr > 0) & (corr < 0.2)).all().all())
corr.loc["September full file"] = {"cont2": 0.142, "cont7": 0.120, "cont3": 0.111}
corr.round(3)

PASS: cont2, cont7, cont3 stay weak and positive in every fold (0 < r < 0.2)


,cont2,cont3,cont7
fold 0 fit portion,0.142,0.108,0.116
fold 1 fit portion,0.143,0.111,0.119
fold 2 fit portion,0.142,0.111,0.121
fold 3 fit portion,0.143,0.110,0.121
fold 4 fit portion,0.142,0.109,0.117
September full file,0.142,0.111,0.120


## 5 Target transformation

**Decision: train Approach 1 on `log1p(loss)`, then convert predictions back to `loss` units before scoring.**

- **MAE rewards the median.** The best constant for absolute error is the median. Least squares on raw `loss` estimates a mean, which the long right tail pulls upward.
- **The log scale is nearly symmetric** (skew about 0.1 in every fold), so converting a log-scale fit back with `expm1` gives a median-type estimate.
- **Predictions stay positive** after converting back.

Evidence without fitting a model: three constants are learned on each fit portion and scored on its held-out fold, in `loss` units. These are training-only CV scores, not validation results, and they are saved as the fold-matched baselines for Gate 3.

In [9]:
cv_rows = []
for k, fit, held_out in iter_folds(folds):
    y_fit, y_out = loss.loc[fit].to_numpy(), loss.loc[held_out].to_numpy()
    for name, c in {"mean": y_fit.mean(), "median": np.median(y_fit),
                    "expm1(mean of log1p)": np.expm1(np.log1p(y_fit).mean())}.items():
        cv_rows.append({"fold": k, "constant": name, "value": c, "claims": len(y_out),
                        "abs_error_sum": np.abs(y_out - c).sum()})
cv = pd.DataFrame(cv_rows)
cv["MAE"] = cv.abs_error_sum / cv.claims
by_fold = cv.pivot(index="fold", columns="constant", values="MAE")

check("Log-scale constant beats the mean constant in every fold", (by_fold["expm1(mean of log1p)"] < by_fold["mean"]).all())
check("Median beats the mean constant in every fold", (by_fold["median"] < by_fold["mean"]).all())

cv[["fold", "constant", "value", "claims", "MAE"]].to_csv(
    FOLD_DIR / "cv_constant_baselines.csv", index=False, float_format="%.10g", lineterminator="\n")

pd.DataFrame({"pooled out-of-fold MAE": cv.groupby("constant").abs_error_sum.sum() / len(loss),
              "sd across folds": by_fold.std(ddof=1)}).round(2).sort_values("pooled out-of-fold MAE")

PASS: Log-scale constant beats the mean constant in every fold
PASS: Median beats the mean constant in every fold


,pooled out-of-fold MAE,sd across folds
constant,,
median,1808.99,15.38
expm1(mean of log1p),1809.78,14.92
mean,1965.90,8.56


### Inverse rule and bad predictions

`LogTargetRegressor` wraps the model so this rule is applied every time:

1. The model predicts `z`, the predicted `log1p(loss)`.
2. **Non-finite `z` (NaN or infinity): the run fails with an error** and is logged as failed. Predictions are never dropped or filled.
3. `z` is clipped to `[log1p(min loss), log1p(max loss)]`, both **learned from the rows the model is fit on**.
4. `prediction = expm1(z)`, so **every prediction is positive and finite**; no negative prediction can reach the evaluator.

No bias correction is applied, since it would move predictions away from the median that MAE rewards. The tests use fake models, so they check the rule, not accuracy.

In [10]:
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.linear_model import Ridge
from sklearn.utils.validation import check_is_fitted

class LogTargetRegressor(RegressorMixin, BaseEstimator):
    """Fit `regressor` on log1p(loss); predict in original loss units (see rule above)."""

    def __init__(self, regressor=None):
        self.regressor = regressor

    def fit(self, X, y):
        y = np.asarray(y, dtype=np.float64)
        if y.ndim != 1 or not np.isfinite(y).all() or (y <= 0).any():
            raise ValueError("loss must be finite and strictly positive")
        self.log_lower_ = float(np.log1p(y.min()))
        self.log_upper_ = float(np.log1p(y.max()))
        self.regressor_ = clone(self.regressor if self.regressor is not None else Ridge())
        self.regressor_.fit(X, np.log1p(y))
        return self

    def predict(self, X):
        check_is_fitted(self, ["regressor_", "log_lower_", "log_upper_"])
        z = np.asarray(self.regressor_.predict(X), dtype=np.float64)
        if not np.isfinite(z).all():
            raise ValueError(f"{int((~np.isfinite(z)).sum())} non-finite model outputs; run failed")
        return np.expm1(np.clip(z, self.log_lower_, self.log_upper_))

In [11]:
class FixedOutput(RegressorMixin, BaseEstimator):
    """Fake model for testing: returns preset log-scale outputs."""
    def __init__(self, outputs=None):
        self.outputs = outputs
    def fit(self, X, y):
        self.fitted_y_ = np.asarray(y)
        return self
    def predict(self, X):
        return np.asarray(self.outputs if self.outputs is not None else self.fitted_y_, dtype=float)

def raises(action):
    try:
        action()
    except ValueError:
        return True
    return False

y = np.array([500.0, 2000.0, 9000.0])
X0 = np.zeros((3, 1))

check("Round trip returns the original losses", np.allclose(LogTargetRegressor(FixedOutput()).fit(X0, y).predict(X0), y))
out = LogTargetRegressor(FixedOutput([-50.0, 0.0, 700.0])).fit(X0, y).predict(X0)
check("Extreme outputs clipped to the fit rows' loss range", np.allclose(out, [500.0, 500.0, 9000.0]))
check("Every prediction finite and positive", np.isfinite(out).all() and (out > 0).all())
for bad in (np.nan, np.inf):
    check(f"Non-finite output ({bad}) fails the run",
          raises(lambda: LogTargetRegressor(FixedOutput([1.0, bad, 2.0])).fit(X0, y).predict(X0)))
check("Zero or negative training loss rejected",
      raises(lambda: LogTargetRegressor(FixedOutput()).fit(X0, np.array([0.0, 10.0, 20.0]))))
a = LogTargetRegressor(FixedOutput()).fit(X0[:2], y[:2])
b = LogTargetRegressor(FixedOutput()).fit(X0[1:], y[1:])
check("Clipping bounds learned separately for each fit", a.log_upper_ < b.log_upper_ and a.log_lower_ < b.log_lower_)

PASS: Round trip returns the original losses
PASS: Extreme outputs clipped to the fit rows' loss range
PASS: Every prediction finite and positive
PASS: Non-finite output (nan) fails the run
PASS: Non-finite output (inf) fails the run
PASS: Zero or negative training loss rejected
PASS: Clipping bounds learned separately for each fit


## 6 GLM preprocessing

The next cell is **copied unchanged from Hector's `gate2_preprocessing.ipynb`** (his schema check, rare-level grouper and safe one-hot encoder), so this notebook can build the same pipeline without importing anything. The cell after it assembles the pipeline exactly as his notebook does and checks it gives his numbers: 808 columns and 352 rare levels across 30 fields.

In [12]:
from scipy.sparse import csr_matrix
from sklearn.base import TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def validate_feature_schema(X, expected_columns, continuous_columns):
    if not isinstance(X, pd.DataFrame):
        raise TypeError("Inputs must be a pandas DataFrame.")

    if not X.columns.is_unique:
        raise ValueError("Duplicate column names are not allowed.")

    missing = sorted(set(expected_columns) - set(X.columns))
    unexpected = sorted(set(X.columns) - set(expected_columns))

    if missing or unexpected:
        raise ValueError(
            f"Schema mismatch. Missing: {missing}; "
            f"unexpected: {unexpected}"
        )

    for col in continuous_columns:
        if not pd.api.types.is_numeric_dtype(X[col]):
            raise ValueError(f"{col} must contain numeric values.")

    continuous_values = X[continuous_columns].to_numpy(
        dtype="float64",
        na_value=np.nan,
    )

    if not np.isfinite(continuous_values).all():
        raise ValueError(
            "Continuous predictors contain missing or infinite values."
        )

class FeatureSchemaValidator(BaseEstimator, TransformerMixin):
    def __init__(self, expected_columns, continuous_columns):
        self.expected_columns = expected_columns
        self.continuous_columns = continuous_columns

    def fit(self, X, y=None):
        validate_feature_schema(
            X, self.expected_columns, self.continuous_columns
        )

        self.feature_names_in_ = X.columns.to_numpy(copy=True)
        self.n_features_in_ = X.shape[1]

        return self

    def transform(self, X):
        check_is_fitted(self, ["feature_names_in_"])

        validate_feature_schema(
            X, self.expected_columns, self.continuous_columns
        )

        return X.loc[:, self.expected_columns].copy()

class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    def __init__(self, min_count=30):
        self.min_count = min_count

    def fit(self, X, y=None):
        if X.eq("__rare__").any().any():
            raise ValueError("The reserved label '__rare__' already exists.")

        self.feature_names_in_ = X.columns.to_numpy(copy=True)
        self.n_features_in_ = X.shape[1]

        self.known_levels_ = {}
        self.rare_levels_ = {}

        for col in self.feature_names_in_:
            counts = X[col].value_counts(dropna=True)

            self.known_levels_[col] = counts.index.tolist()
            self.rare_levels_[col] = counts.index[
                counts < self.min_count
            ].tolist()

        return self

    def transform(self, X):
        check_is_fitted(self, ["known_levels_", "rare_levels_"])

        if list(X.columns) != list(self.feature_names_in_):
            raise ValueError(
                "Categorical columns or their order differ from fitting."
            )

        transformed = X.astype("string").copy()

        for col in self.feature_names_in_:
            rare = self.rare_levels_[col]

            if rare:
                is_rare = transformed[col].isin(rare)
                is_unseen = ~transformed[col].isin(
                    self.known_levels_[col]
                )

                transformed[col] = transformed[col].mask(
                    is_rare | is_unseen,
                    "__rare__",
                )

            transformed[col] = transformed[col].fillna("__unseen__")

        return transformed

class PlaceholderSafeOneHotEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.feature_names_in_ = X.columns.to_numpy(copy=True)
        self.n_features_in_ = X.shape[1]

        self.active_columns_ = []
        categories = []

        for col in self.feature_names_in_:
            levels = sorted(
                label
                for label in X[col].unique()
                if label != "__unseen__"
            )

            if levels:
                self.active_columns_.append(col)
                categories.append(levels)

        self.encoder_ = None

        if self.active_columns_:
            self.encoder_ = OneHotEncoder(
                categories=categories,
                handle_unknown="ignore",
                drop=None,
                sparse_output=True,
                dtype="float64",
            )

            self.encoder_.fit(X[self.active_columns_])

        return self

    def transform(self, X):
        check_is_fitted(self, ["active_columns_", "encoder_"])

        if list(X.columns) != list(self.feature_names_in_):
            raise ValueError(
                "Categorical columns or their order differ from fitting."
            )

        if self.encoder_ is None:
            return csr_matrix((len(X), 0), dtype="float64")

        return self.encoder_.transform(X[self.active_columns_])

In [13]:
def build_glm_preprocessing(min_count=RARE_THRESHOLD):
    categorical = Pipeline([("rare_grouping", RareCategoryGrouper(min_count=min_count)),
                            ("onehot", PlaceholderSafeOneHotEncoder())])
    columns = ColumnTransformer([("categorical", categorical, CATEGORICAL_COLUMNS),
                                 ("continuous", StandardScaler(), CONTINUOUS_COLUMNS)],
                                remainder="drop", sparse_threshold=1.0)
    return Pipeline([("schema", FeatureSchemaValidator(expected_columns=FEATURE_COLUMNS,
                                                       continuous_columns=CONTINUOUS_COLUMNS)),
                     ("preprocessing", columns)])

types = {"id": "int64", **{c: "string" for c in CATEGORICAL_COLUMNS}, **{c: "float64" for c in CONTINUOUS_COLUMNS}}
X_train = pd.concat(chunk[chunk.id.isin(wanted)] for chunk in
                    pd.read_csv(SOURCE, usecols=["id"] + FEATURE_COLUMNS, dtype=types, chunksize=20_000)
                    ).set_index("id").loc[loss.index, FEATURE_COLUMNS]

full = build_glm_preprocessing().fit(X_train)
grouper = full.named_steps["preprocessing"].named_transformers_["categorical"].named_steps["rare_grouping"]
check("Matches Hector's pipeline: 808 output columns", full.transform(X_train.head(5)).shape[1] == 808)
check("Matches Hector's pipeline: 352 rare levels in 30 fields",
      sum(map(len, grouper.rare_levels_.values())) == 352 and sum(map(bool, grouper.rare_levels_.values())) == 30)

PASS: Matches Hector's pipeline: 808 output columns
PASS: Matches Hector's pipeline: 352 rare levels in 30 fields


## 7 Approach 1 configuration

The configuration is written to `artifacts/october/gate 2/approach1_glm_config.json` so it is frozen before any Gate 3 fit.

| Setting | Value |
| :--- | :--- |
| Preprocessing | Hector's GLM pipeline (rare grouping < 30, one-hot, standardized continuous) |
| Target | `log1p(loss)` with the inverse rule above |
| Model | Ridge regression (L2 penalty), `lsqr` solver |
| Search | 6 values of `alpha`: 1 to 100,000 (30 CV fits, within the team cap of 6 configurations) |
| Selection | Lowest pooled out-of-fold MAE in `loss` units; ties within 1 unit go to the larger `alpha` |
| Validation | Only the selected configuration, scored once in Gate 3 |

In [14]:
config = {
    "config_id": "approach1-glm-v1",
    "status": "PROPOSED scaffold; freeze at Gate 2 review before any Gate 3 fit",
    "owner": "Rachel Cheung",
    "issues": {"scaffold": "#40", "training_and_tuning": "#44"},
    "approach": "Approach 1: regularized linear model (ridge) on log1p(loss)",
    "hypothesis": ("September found weak, mostly additive marginal signals. A penalized linear model on the log "
                   "scale should beat both constant predictors on held-out MAE, but will miss nonlinear effects "
                   "and interactions between fields."),
    "inputs": {"features": "all 130 source predictors (cat1-cat116, cont1-cont14)",
               "excluded": ["id", "loss", "log1p_loss", "partition", "September summaries and bin labels"],
               "split_manifest": metadata["split"]["version"],
               "cv_folds": "artifacts/october/gate 2/cv_folds/october_cv_folds.csv"},
    "preprocessing": {"source": "Hector's GLM pipeline (notebooks/gate2_preprocessing.ipynb)",
                      "rare_min_count": RARE_THRESHOLD,
                      "unseen_or_missing_category": "rare column if the field has one, otherwise all zeros",
                      "continuous": "StandardScaler fit on the fit rows",
                      "correlated_pairs": "cont11/cont12 and cont1/cont9 retained; L2 penalty shares weight",
                      "fit_scope": "refit inside every CV fit portion and for the final training refit"},
    "target": {"transform": "log1p(loss)",
               "inverse": "expm1(clip(z, log1p(min fit loss), log1p(max fit loss)))",
               "non_finite_output": "raise error; log the run as failed; never drop or fill predictions",
               "bias_correction": "none (median-type estimate suits MAE)"},
    "model": {"estimator": "sklearn.linear_model.Ridge", "solver": "lsqr", "tol": 1e-06, "fit_intercept": True},
    "search_space": {"alpha": [1.0, 10.0, 100.0, 1000.0, 10000.0, 100000.0],
                     "complexity_order": "larger alpha = stronger penalty = simpler model"},
    "evaluation": {"cv_fits": "6 configurations x 5 folds = 30",
                   "selection_metric": "pooled out-of-fold MAE in original loss units",
                   "also_report": "each fold's MAE and claim count, mean and sample sd, fold-matched mean/median constants",
                   "tie_rule": "within 1 loss unit of the best pooled MAE counts as tied; pick the larger alpha",
                   "edge_rule": "if the best alpha is at either end of the grid, record it; extending needs a team amendment",
                   "final_refit": "refit the selected alpha on all 141,238 training claims",
                   "validation_release": "the single selected candidate only, scored once with the shared evaluator",
                   "test_partition": "never used in October"},
    "environment": {"python": sys.version.split()[0],
                    **{p: version(p) for p in ["pandas", "numpy", "scipy", "scikit-learn"]}},
}
CONFIG_FILE.write_text(json.dumps(config, indent=2) + "\n")
check("Search space has at most 6 configurations (team cap)", len(config["search_space"]["alpha"]) <= 6)

def build_glm_pipeline(alpha):
    if alpha not in config["search_space"]["alpha"]:
        raise ValueError(f"alpha={alpha} is not in the frozen search space")
    m = config["model"]
    ridge = Ridge(alpha=alpha, solver=m["solver"], tol=m["tol"], fit_intercept=m["fit_intercept"])
    return Pipeline([("preprocessing", build_glm_preprocessing(config["preprocessing"]["rare_min_count"])),
                     ("model", LogTargetRegressor(regressor=ridge))])

check("Alpha outside the frozen grid rejected", raises(lambda: build_glm_pipeline(0.5)))

PASS: Search space has at most 6 configurations (team cap)
PASS: Alpha outside the frozen grid rejected


**Smoke test.** Every configuration is fit on 2,000 training claims with **artificial** targets, so this checks the plumbing only. It says nothing about accuracy and uses none of the tuning budget.

In [15]:
fit_ids = folds.loc[folds.fold.ne(0), "id"].head(2000)
out_ids = folds.loc[folds.fold.eq(0), "id"].head(500)
X_fit, X_out = X_train.loc[fit_ids], X_train.loc[out_ids]
artificial_y = np.random.default_rng(42).lognormal(mean=7.6, sigma=0.8, size=len(X_fit))   # NOT real loss

pipelines = {}
for alpha in config["search_space"]["alpha"]:
    pipelines[alpha] = build_glm_pipeline(alpha).fit(X_fit, artificial_y)
    pred = pipelines[alpha].predict(X_out)
    check(f"alpha={alpha:g}: one finite, positive prediction per claim",
          pred.shape == (len(X_out),) and np.isfinite(pred).all() and (pred > 0).all())

pipe = pipelines[1000.0]
unseen = X_out.head(len(CATEGORICAL_COLUMNS)).copy()
for row, col in enumerate(CATEGORICAL_COLUMNS):
    unseen.iloc[row, unseen.columns.get_loc(col)] = f"__made_up_{col}__"
pred_unseen = pipe.predict(unseen)
check("Made-up labels in all 116 fields still give finite, positive predictions",
      np.isfinite(pred_unseen).all() and (pred_unseen > 0).all())

original = pd.Series(pipe.predict(X_out), index=X_out.index)
shuffled = X_out.sample(frac=1, random_state=7)
check("Predictions stay matched to claim IDs after shuffling",
      np.allclose(pd.Series(pipe.predict(shuffled), index=shuffled.index).loc[original.index], original))

PASS: alpha=1: one finite, positive prediction per claim
PASS: alpha=10: one finite, positive prediction per claim
PASS: alpha=100: one finite, positive prediction per claim
PASS: alpha=1000: one finite, positive prediction per claim
PASS: alpha=10000: one finite, positive prediction per claim
PASS: alpha=100000: one finite, positive prediction per claim
PASS: Made-up labels in all 116 fields still give finite, positive predictions
PASS: Predictions stay matched to claim IDs after shuffling


## 8 Summary

In [16]:
print("Folds:", FOLD_FILE.relative_to(ROOT), "| SHA-256", sha256(FOLD_FILE)[:12])
print("Held-out sizes:", folds.fold.value_counts().sort_index().tolist())
print("Training p95 for Gate 3 high-loss slices:", round(np.quantile(loss, 0.95), 2))
print("Target: log1p(loss); inverse expm1 with fit-range clipping; non-finite output fails the run")
print("Config:", CONFIG_FILE.relative_to(ROOT), "| alphas", config["search_space"]["alpha"])
print("Validation targets read: 0 | test targets read: 0 | real-target GLM fits: 0")

passed = sum(ok for _, ok in results)
print(f"\n{passed} of {len(results)} checks passed")
assert passed == len(results), "A check failed; see FAIL lines above."

Folds: artifacts/october/gate 2/cv_folds/october_cv_folds.csv | SHA-256 82fb7f3d9899
Held-out sizes: [28248, 28248, 28248, 28247, 28247]
Training p95 for Gate 3 high-loss slices: 8511.92
Target: log1p(loss); inverse expm1 with fit-range clipping; non-finite output fails the run
Config: artifacts/october/gate 2/approach1_glm_config.json | alphas [1.0, 10.0, 100.0, 1000.0, 10000.0, 100000.0]
Validation targets read: 0 | test targets read: 0 | real-target GLM fits: 0

36 of 36 checks passed
